# 01 — Euler–Bernoulli Cantilever Baseline

The **classical, ML-free reference solver** that the NeuralBeam GNN surrogate is
trained against (see the paid Gumroad release for the full ML pipeline).

We solve a tip-loaded cantilever of length `L` two ways and compare them:

1. **Analytical Euler–Bernoulli** closed form
   $$ w(x) = \frac{P x^2 (3L - x)}{6 E I} $$
2. **Finite element method** using 2-node Hermite beam elements (4 DOF per
   element: transverse displacement + rotation at each node).

We then extract the same features the downstream VAE / GNN / PINN models
consume — nodal displacements, curvature, bending moments, strain energy —
and save them to disk as a reference baseline.

Pure NumPy + SciPy + Matplotlib. No PyTorch.

## 1. Import required libraries

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Define beam parameters

Steel cantilever, rectangular cross-section. The same constants used by the
NeuralBeam paid release (`L = 300 mm`, `T = 10 mm`, `E = 210 GPa`) so the
features extracted here are directly comparable with the ML pipeline.

In [ ]:
L = 300.0          # span [mm]
T = 10.0           # out-of-plane thickness (width) [mm]
H = 30.0           # uniform section height [mm]
E = 210_000.0      # Young's modulus [MPa]
P = 1_000.0        # tip load [N]

I = T * H**3 / 12.0
print(f"Second moment of area  I = {I:.2f}  mm^4")
print(f"Flexural rigidity     EI = {E*I:.3e}  N·mm^2")

## 3. Euler–Bernoulli analytical solution

For a uniform cantilever with a tip load `P`, the closed-form deflection is

$$ w(x) = \frac{P\,x^2\,(3L - x)}{6\,E\,I} $$

with tip deflection

$$ \delta_{\text{tip}} = \frac{P L^3}{3 E I} $$

This is our ground-truth reference.

In [ ]:
x_fine     = np.linspace(0, L, 401)
w_analyt   = P * x_fine**2 * (3*L - x_fine) / (6 * E * I)
delta_tip  = P * L**3 / (3 * E * I)
print(f"Analytical tip deflection : {delta_tip:.4f} mm")

## 4. Discretize the beam

Divide the span into `N_EL = 40` Hermite beam elements (41 nodes), 2 DOF per
node: transverse displacement `w` and rotation `θ`. Total DOF count =
`2 × n_nodes`.

In [ ]:
N_EL    = 40
n_nodes = N_EL + 1
n_dof   = 2 * n_nodes

x_nodes = np.linspace(0, L, n_nodes)
le      = L / N_EL                       # element length (uniform)
print(f"{N_EL} elements, {n_nodes} nodes, {n_dof} DOFs, le = {le:.3f} mm")

## 5. Assemble the global stiffness matrix

The classical 4×4 Hermite element stiffness for a beam of length $\ell$:

$$
k_e = \frac{EI}{\ell^3}
\begin{bmatrix}
 12 &  6\ell & -12 &  6\ell \\
 6\ell & 4\ell^2 & -6\ell & 2\ell^2 \\
-12 & -6\ell &  12 & -6\ell \\
 6\ell & 2\ell^2 & -6\ell & 4\ell^2
\end{bmatrix}
$$

DOF ordering per element: $[w_1, \theta_1, w_2, \theta_2]$.

In [ ]:
def element_stiffness(EI, le):
    c = EI / le**3
    return c * np.array([
        [ 12,    6*le,   -12,    6*le  ],
        [  6*le, 4*le**2, -6*le, 2*le**2],
        [-12,   -6*le,    12,   -6*le  ],
        [  6*le, 2*le**2, -6*le, 4*le**2],
    ])

K = np.zeros((n_dof, n_dof))
ke = element_stiffness(E*I, le)
for e in range(N_EL):
    dofs = [2*e, 2*e+1, 2*e+2, 2*e+3]
    for i in range(4):
        for j in range(4):
            K[dofs[i], dofs[j]] += ke[i, j]

print(f"K shape : {K.shape}")
print(f"K symmetric? {np.allclose(K, K.T)}")

## 6. Apply boundary conditions and loads

Cantilever fixed at `x = 0` → constrain DOFs `0` (`w`) and `1` (`θ`).
Point load `P` applied at the last node's transverse DOF.

In [ ]:
F = np.zeros(n_dof)
F[2 * (n_nodes - 1)] = -P            # tip transverse DOF; negative = downward

fixed_dofs = [0, 1]
free_dofs  = [d for d in range(n_dof) if d not in fixed_dofs]
print(f"{len(fixed_dofs)} fixed DOFs, {len(free_dofs)} free DOFs")

## 7. Solve the FEA system

Partition $KU = F$ to the free DOFs and solve.

In [ ]:
K_ff = K[np.ix_(free_dofs, free_dofs)]
F_f  = F[free_dofs]

U_f  = la.solve(K_ff, F_f, assume_a="pos")

U = np.zeros(n_dof)
U[free_dofs] = U_f

w_fea     = -U[0::2]                  # flip sign so downward load -> positive deflection
theta_fea = -U[1::2]
print(f"FEA tip deflection : {w_fea[-1]:.4f} mm")

## 8. Compare FEA vs analytical

Hermite beam elements are **exact** for cubic deflection fields, and the
Euler–Bernoulli analytical solution for a point-loaded uniform cantilever is
cubic — so FEA and analytical should agree to round-off.

In [ ]:
w_at_nodes_analyt = P * x_nodes**2 * (3*L - x_nodes) / (6 * E * I)

err          = w_fea - w_at_nodes_analyt
l2_err       = np.linalg.norm(err) / np.linalg.norm(w_at_nodes_analyt)
max_rel_err  = np.max(np.abs(err[1:]) / np.abs(w_at_nodes_analyt[1:]))
print(f"L2 relative error : {l2_err:.3e}")
print(f"Max rel. error    : {max_rel_err:.3e}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x_fine,  w_analyt, "-",  color="seagreen", lw=2, label="analytical")
ax.plot(x_nodes, w_fea,    "o",  color="crimson", ms=4, label="FEA nodes")
ax.set_xlabel("x [mm]"); ax.set_ylabel("w(x) [mm]")
ax.set_title("Deflection — analytical vs FEA")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 9. Extract features for the ML pipeline

The NeuralBeam paid release trains its GNN against these same per-node fields:

| Feature | Formula |
|---|---|
| Deflection `w(x)` | nodal displacement |
| Rotation `θ(x)` | nodal rotation |
| Curvature `κ(x)` | `d²w/dx²` (finite diff) |
| Bending moment `M(x)` | `EI · κ` (also `−P·(L−x)` analytically) |
| Shear force `V(x)` | `dM/dx` (constant `−P` here) |
| Max-fibre stress `σ(x)` | `|M|·c / I`, `c = H/2` |
| Strain energy density `U(x)` | `M² / (2 EI)` |

In [ ]:
curvature      = np.gradient(np.gradient(w_fea, x_nodes), x_nodes)
M_fea          = E * I * curvature
M_analyt       = -P * (L - x_nodes)
shear_fea      = np.gradient(M_fea, x_nodes)
sigma_max      = np.abs(M_fea) * (H / 2.0) / I
strain_energy  = M_fea**2 / (2.0 * E * I)
U_total        = np.trapz(strain_energy, x_nodes)

print(f"Total strain energy : {U_total:.3f} N·mm")
print(f"Peak |sigma|        : {np.abs(sigma_max).max():.2f} MPa  "
      f"(analytical = {P*L*(H/2)/I:.2f} MPa)")

## 10. Visualize deflection, bending moment, shear

Three-panel summary plot.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

axes[0].plot(x_nodes, w_fea, "o-", color="seagreen", ms=3)
axes[0].set_ylabel("w(x) [mm]"); axes[0].set_title("Deflection")
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_nodes, M_fea,    "o-", color="crimson", ms=3, label="FEA (EI·κ)")
axes[1].plot(x_nodes, M_analyt, "--", color="black",   lw=1, label="analytical −P(L−x)")
axes[1].set_ylabel("M(x) [N·mm]"); axes[1].set_title("Bending moment")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(x_nodes, shear_fea, "o-", color="navy", ms=3)
axes[2].axhline(-P, color="black", ls="--", lw=1, label=f"−P = {-P:.0f} N")
axes[2].set_xlabel("x [mm]"); axes[2].set_ylabel("V(x) [N]"); axes[2].set_title("Shear force")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 11. Save baseline results

Dump features + metadata to a compressed `.npz` for downstream notebooks
(or to compare against ML-surrogate predictions from the paid release).

In [ ]:
out_dir  = Path("results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "baseline_uniform_cantilever.npz"

np.savez_compressed(
    out_path,
    x_nodes=x_nodes,
    w=w_fea,
    theta=theta_fea,
    curvature=curvature,
    M=M_fea,
    V=shear_fea,
    sigma_max=sigma_max,
    strain_energy=strain_energy,
    meta=dict(L=L, T=T, H=H, E=E, P=P, I=I, n_el=N_EL),
)
print(f"Saved -> {out_path.resolve()}")